<a href="https://colab.research.google.com/github/yasirusman85/LeNet-5/blob/main/Yasir_LeNet5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LeNet-5 CNN Model for MNIST Handwritten Digit Recognition

This section implements the LeNet-5 convolutional neural network (CNN) model to classify handwritten digits from the MNIST dataset. The process involves:

1.  **Loading and Preprocessing Data**: Loading the MNIST dataset, padding images from 28x28 to 32x32, normalizing pixel values, and converting labels to one-hot encoding.
2.  **Building LeNet-5 Architecture**: Constructing the LeNet-5 model with three convolutional layers, two average pooling layers, and two dense layers.
3.  **Training the Model**: Training the model for 10 epochs.
4.  **Evaluating and Reporting Accuracy**: Assessing the model's performance on the test set and reporting the test accuracy.

In [1]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras import datasets, layers, models, utils
import numpy as np

# Load the MNIST dataset
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

print(f"Original training images shape: {train_images.shape}")
print(f"Original test images shape: {test_images.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Original training images shape: (60000, 28, 28)
Original test images shape: (10000, 28, 28)


### Data Preprocessing

1.  **Padding Images**: LeNet-5 typically expects 32x32 input images. The MNIST images are 28x28, so we'll pad them with zeros to 32x32.
2.  **Reshaping**: Reshape images to include a channel dimension (e.g., `(batch_size, 32, 32, 1)` for grayscale).
3.  **Normalization**: Normalize pixel values to the range [0, 1].
4.  **One-Hot Encoding**: Convert integer labels to one-hot encoded vectors.

In [2]:
# Pad images from 28x28 to 32x32
train_images_padded = np.pad(train_images, ((0,0),(2,2),(2,2)), 'constant')
test_images_padded = np.pad(test_images, ((0,0),(2,2),(2,2)), 'constant')

# Reshape images to add a channel dimension (grayscale, 1 channel)
train_images_reshaped = train_images_padded.reshape((60000, 32, 32, 1)).astype('float32')
test_images_reshaped = test_images_padded.reshape((10000, 32, 32, 1)).astype('float32')

# Normalize pixel values to be between 0 and 1
train_images_normalized = train_images_reshaped / 255.0
test_images_normalized = test_images_reshaped / 255.0

# Convert labels to one-hot encoding
train_labels_one_hot = utils.to_categorical(train_labels, 10)
test_labels_one_hot = utils.to_categorical(test_labels, 10)

print(f"Padded and reshaped training images shape: {train_images_normalized.shape}")
print(f"Padded and reshaped test images shape: {test_images_normalized.shape}")
print(f"One-hot encoded training labels shape: {train_labels_one_hot.shape}")

Padded and reshaped training images shape: (60000, 32, 32, 1)
Padded and reshaped test images shape: (10000, 32, 32, 1)
One-hot encoded training labels shape: (60000, 10)


### LeNet-5 Model Architecture

The LeNet-5 architecture consists of the following layers:

1.  **C1 (Convolutional Layer)**: 6 filters of size 5x5, stride 1, tanh activation.
2.  **S2 (Average Pooling Layer)**: 6 filters of size 2x2, stride 2.
3.  **C3 (Convolutional Layer)**: 16 filters of size 5x5, stride 1, tanh activation.
4.  **S4 (Average Pooling Layer)**: 16 filters of size 2x2, stride 2.
5.  **C5 (Convolutional Layer)**: 120 filters of size 5x5, stride 1, tanh activation (this acts like a fully connected layer since the input feature map size is 1x1).
6.  **Flatten Layer**: Flattens the output of C5.
7.  **F6 (Fully Connected Layer)**: 84 units, tanh activation.
8.  **Output Layer**: 10 units (for 10 classes), softmax activation.

In [3]:
# Build the LeNet-5 model
model = models.Sequential([
    # C1: Convolutional layer
    layers.Conv2D(filters=6, kernel_size=(5, 5), activation='tanh', input_shape=(32, 32, 1)),
    # S2: Average pooling layer
    layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),
    # C3: Convolutional layer
    layers.Conv2D(filters=16, kernel_size=(5, 5), activation='tanh'),
    # S4: Average pooling layer
    layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),
    # C5: Convolutional layer (acting as fully connected)
    layers.Conv2D(filters=120, kernel_size=(5, 5), activation='tanh'),
    # Flatten the output for the fully connected layers
    layers.Flatten(),
    # F6: Fully connected layer
    layers.Dense(units=84, activation='tanh'),
    # Output layer
    layers.Dense(units=10, activation='softmax')
])

# Display the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Compile and Train the Model

We will compile the model using the Adam optimizer and categorical cross-entropy loss, and then train it for 10 epochs.

In [4]:
# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(train_images_normalized, train_labels_one_hot,
                    epochs=10,
                    validation_data=(test_images_normalized, test_labels_one_hot),
                    verbose=1)

print("Model training complete.")

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 40s 20ms/step - accuracy: 0.9286 - loss: 0.2314 - val_accuracy: 0.9652 - val_loss: 0.1132
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 36s 19ms/step - accuracy: 0.9740 - loss: 0.0846 - val_accuracy: 0.9788 - val_loss: 0.0700
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 36s 19ms/step - accuracy: 0.9830 - loss: 0.0564 - val_accuracy: 0.9817 - val_loss: 0.0570
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - accuracy: 0.9863 - loss: 0.0444 - val_accuracy: 0.9860 - val_loss: 0.0443
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - accuracy: 0.9891 - loss: 0.0345 - val_accuracy: 0.9849 - val_loss: 0.0543
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 37s 20ms/step - accuracy: 0.9907 - loss: 0.0293 - val_accuracy: 0.9863 - val_loss: 0.0432
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - accuracy: 0.9918 - loss: 0.0253 - val_accuracy: 0.9867 - val_loss: 0.0451
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 41s 20ms/step - accuracy: 0.9931 -

### Evaluate Model Performance

Finally, we will evaluate the trained model on the test dataset to report its accuracy.

In [5]:
# Evaluate the model on the test dataset
loss, accuracy = model.evaluate(test_images_normalized, test_labels_one_hot, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.0437
Test Accuracy: 0.9862


### Saving Model and Metrics for Frontend

To prepare for our Streamlit frontend, we need to save the trained LeNet-5 model and the evaluation metrics (test accuracy and loss) so that the separate application can load and display them. We will save the model in Keras's native format and the metrics in a JSON file.

In [6]:
# Save the trained model
model.save('lenet5_mnist_model.h5')
print("Model saved as 'lenet5_mnist_model.h5'")

# Save the metrics (accuracy and loss)
import json
metrics = {'accuracy': accuracy, 'loss': loss}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f)
print("Metrics saved as 'metrics.json'")

Model saved as 'lenet5_mnist_model.h5'
Metrics saved as 'metrics.json'
